# Gold Layer — Aggregated & Business-Ready

Aggregate Silver data into business-ready tables that answer key questions.

- **Source:** `silver_vehicle_registrations` Delta table
- **Output:** Multiple Gold Delta tables
- **Rule:** Aggregations and metrics only — no row-level cleaning.


In [1]:
from pyspark.sql import functions as F

SILVER_TABLE = "silver_vehicle_registrations"

In [2]:
df_silver = spark.table(SILVER_TABLE)

print(f"Silver row count: {df_silver.count()}")
df_silver.printSchema()

Silver row count: 938322
root
 |-- registration_date: date (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- colour: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



## 1. Monthly Registrations by Manufacturer
Which car makers dominate registrations, and how do they trend monthly?


In [3]:
df_monthly_manufacturer = (
    df_silver
    .groupBy("year", "month", "manufacturer")
    .agg(F.count("*").alias("total_registrations"))
    .orderBy("year", "month", F.desc("total_registrations"))
)

df_monthly_manufacturer.write.format("delta").mode("overwrite").saveAsTable("gold_monthly_by_manufacturer")

print("Written: gold_monthly_by_manufacturer")
df_monthly_manufacturer.show(10)

Written: gold_monthly_by_manufacturer
+----+-----+-------------+-------------------+
|year|month| manufacturer|total_registrations|
+----+-----+-------------+-------------------+
|2025|    1|      Perodua|              23245|
|2025|    1|       Proton|               9688|
|2025|    1|       Toyota|               7570|
|2025|    1|        Honda|               3638|
|2025|    1|        Chery|               1678|
|2025|    1|   Mitsubishi|               1227|
|2025|    1|Mercedes Benz|                752|
|2025|    1|        Mazda|                707|
|2025|    1|          BMW|                601|
|2025|    1|       Nissan|                578|
+----+-----+-------------+-------------------+
only showing top 10 rows


## 2. EV Registration Trend
How are electric vehicle registrations growing over time?


In [ ]:
df_ev_trend = (
    df_silver
    .filter(F.col("fuel_type") == "electric")
    .groupBy("year", "month")
    .agg(F.count("*").alias("ev_registrations"))
    .orderBy("year", "month")
)

df_ev_trend.write.format("delta").mode("overwrite").saveAsTable("gold_ev_registrations_trend")

print("Written: gold_ev_registrations_trend")
df_ev_trend.show(10)

Written: gold_ev_registration_trend
+----+-----+----------------+
|year|month|ev_registrations|
+----+-----+----------------+
|2025|    1|            1691|
|2025|    2|            2160|
|2025|    3|            2976|
|2025|    4|            2892|
|2025|    5|            4152|
|2025|    6|            3272|
|2025|    7|            2792|
|2025|    8|            3461|
|2025|    9|            3532|
|2025|   10|            4345|
+----+-----+----------------+
only showing top 10 rows


## 3. Registration Volumes by State
Which states have the most vehicle registrations?


In [5]:
df_state = (
    df_silver
    .groupBy("state")
    .agg(F.count("*").alias("total_registrations"))
    .orderBy(F.desc("total_registrations"))
)

df_state.write.format("delta").mode("overwrite").saveAsTable("gold_state_registrations")

print("Written: gold_state_registrations")
df_state.show()

Written: gold_state_registrations
+-----------------+-------------------+
|            state|total_registrations|
+-----------------+-------------------+
|      Rakan Niaga|             809785|
|W.P. Kuala Lumpur|              46262|
|         Selangor|              16304|
|            Johor|              14759|
|            Sabah|              12289|
|          Sarawak|              11138|
|     Pulau Pinang|               6527|
|            Perak|               4262|
|            Kedah|               4215|
|           Pahang|               3207|
|         Kelantan|               2709|
|           Melaka|               2516|
|  Negeri Sembilan|               1887|
|       Terengganu|               1794|
|           Perlis|                668|
+-----------------+-------------------+



## 4. Yearly Market Share by Manufacturer
How has each manufacturer's share of total registrations changed year over year?


In [6]:
# Count registrations per manufacturer per year
df_yearly = (
    df_silver
    .groupBy("year", "manufacturer")
    .agg(F.count("*").alias("registrations"))
)

# Calculate each manufacturer's share of the year's total
from pyspark.sql.window import Window

w = Window.partitionBy("year")

df_market_share = (
    df_yearly
    .withColumn("yearly_total", F.sum("registrations").over(w))
    .withColumn("market_share_pct", F.round(F.col("registrations") / F.col("yearly_total") * 100, 2))
    .drop("yearly_total")
    .orderBy("year", F.desc("market_share_pct"))
)

df_market_share.write.format("delta").mode("overwrite").saveAsTable("gold_yearly_market_share")

print("Written: gold_yearly_market_share")
df_market_share.show(10)

Written: gold_yearly_market_share
+----+-------------+-------------+----------------+
|year| manufacturer|registrations|market_share_pct|
+----+-------------+-------------+----------------+
|2025|      Perodua|       359904|           41.35|
|2025|       Proton|       151561|           17.41|
|2025|       Toyota|       129085|           14.83|
|2025|        Honda|        75599|            8.69|
|2025|        Chery|        31666|            3.64|
|2025|          BYD|        14407|            1.66|
|2025|   Mitsubishi|        13856|            1.59|
|2025|        Mazda|         9589|             1.1|
|2025|Mercedes Benz|         8976|            1.03|
|2025|          BMW|         7873|             0.9|
+----+-------------+-------------+----------------+
only showing top 10 rows


## Verify all Gold tables


In [8]:
gold_tables = [
    "gold_monthly_by_manufacturer",
    "gold_ev_registrations_trend",
    "gold_state_registrations",
    "gold_yearly_market_share",
]

for table in gold_tables:
    count = spark.table(table).count()
    print(f"{table}: {count} rows")

gold_monthly_by_manufacturer: 828 rows
gold_ev_registrations_trend: 13 rows
gold_state_registrations: 15 rows
gold_yearly_market_share: 160 rows
